# 02 — Modelo de posible fraude (V4 + V5)

Entrena los tres artefactos que la API carga al arranque:

1. **LightGBM** (`fraud_lgbm.txt`) — probabilidad supervisada + SHAP top-3.
2. **IsolationForest** (`anomaly_iforest.joblib`) — indicador de anomalía no supervisado.
3. **NearestNeighbors** (`anomaly_knn.joblib`) — siniestro "normal" más parecido.

El cuaderno reutiliza `notebooks._training` (mismo código que `train_all`), de modo que el procedimiento es reproducible desde la CLI o el cuaderno.

**Importante (explicabilidad §2.4):** la probabilidad de la red supervisada NO se suma al score aditivo de reglas; viaja en un campo separado de `ClaimRiskScore` (`ml_probability`) y se muestra al analista en su propio acordeón.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "app").is_dir() and (REPO_ROOT.parent / "app").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"REPO_ROOT = {REPO_ROOT}")

In [ ]:
from notebooks._training import (
    build_dataset,
    train_anomaly,
    train_classifier,
    train_knn,
)

In [ ]:
dataset = build_dataset()
print(f"rows         = {dataset.X.shape[0]}")
print(f"features     = {dataset.X.shape[1]}")
print(f"positive_rate= {dataset.positive_rate:.3f}")

## V4 — LightGBM supervisado

In [ ]:
classifier_report = train_classifier(dataset)
print(f"holdout AUC = {classifier_report.holdout_auc:.3f}")
print(f"CV AUC      = {classifier_report.cv_mean:.3f} ± {classifier_report.cv_std:.3f}")
print(f"saved       → {classifier_report.booster_path}")

## V5 — IsolationForest anomalía + kNN sidecar

In [ ]:
anomaly_report = train_anomaly(dataset)
print(f"score mean = {anomaly_report.score_mean:.3f}")
print(f"score std  = {anomaly_report.score_std:.3f}")
print(f"saved      → {anomaly_report.model_path}")

knn_report = train_knn(dataset)
print(f"normal anchors = {knn_report.anchors}")
print(f"saved          → {knn_report.model_path}")

**Listo.** Los tres artefactos quedan en `data/models/`. Reinicia el backend para que el lifespan los recoja — `/api/v1/status/ai` debe mostrar `fraud_model_present: true` y `anomaly_model_present: true`.

Para métricas y SHAP detallados → ver `03_evaluacion_modelo.ipynb`.